# Group XX - Machine Learning, Element 1: Apply dataset preparation mechanisms
| Item | Details |
|---|---|
| Group | **Group XX** |
| Members | **IGIHOZO Bienvenue** and **MUGISHA Aime Patrick** |
| Question | **Q2 - Next-day rain early warning for MINEMA district disaster offices (own scenario)** |
| Stage | **2 - Define requirements and acquire data** |
| Driver / Navigator | Driver: IGIHOZO Bienvenue / Navigator: MUGISHA Aime Patrick |


## Stage 2 - Define requirements and acquire data (PC 1.2)

### Scenario (summary - full brief in `docs/Q2_Scenario_Brief_and_Proposal.docx`)
MINEMA (Ministry in charge of Emergency Management) works with district disaster management committees. Heavy and repeated
rain triggers floods and landslides - in May 2023 floods and landslides in the Western and Northern provinces (Rubavu, Nyabihu,
Ngororero, Musanze...) killed more than 130 people. Every evening the duty officer must decide whether to **send an SMS
warning and put response teams on standby for tomorrow**. MINEMA asks us to prepare a dataset so that a model can later
answer: **"Will it rain tomorrow at this station?"** (`rain_tomorrow` = 1 / 0, binary classification).

### 2.1 Data requirements
| Requirement | Decision |
|---|---|
| **Target** | `rain_tomorrow` - rain or drizzle observed at the station on the next calendar day (1/0) |
| **Unit of observation** | one weather station on one day |
| **Features needed** | today's temperature, dew point / humidity, min-max temperature, wind, visibility, rain amount, weather events (rain, thunder, fog), season, station/district |
| **Must NOT be used** | anything measured tomorrow (leakage); random split across time (would let the model see the future) |
| **Time period** | at least 5 years of daily data (seasons repeat every year) -> 2015-2025 |
| **Area** | stations near high-risk districts: Rubavu, Musanze, Rusizi + Kigali for comparison |
| **Records needed** | > 1,000 rows, >= 8 columns, numeric + categorical |
| **Licence** | open / free for education; no personal data |

### 2.2 Comparison of possible sources
| Source | Relevance | Size | Licence | Quality | Decision |
|---|---|---|---|---|---|
| **NOAA NCEI Global Summary of the Day (GSOD)** - real station observations | real measurements at Kigali, Kamembe, Gisenyi, Ruhengeri airports | ~10,600 station-days 2015-2025, 28 columns | free, no login (US Gov); WMO Res. 40 - non-commercial use | real problems: missing codes, gaps, empty columns, imperial units | **chosen** |
| Open-Meteo historical weather API (ERA5 reanalysis) | modelled values for any point | unlimited | CC BY 4.0 | complete and smooth (model output, not observation) - **too clean** for this task and not a real observation | rejected |
| Meteo Rwanda station data / NISR | best local source | - | request / registration needed | not downloadable in time | rejected (recommended for the future) |

In [1]:
import sys  # lets us tell Python where our own code (src/) is
from pathlib import Path  # file paths that work on every computer
import warnings  # hide harmless library notices
warnings.filterwarnings("ignore")  # keep output readable
import numpy as np  # maths on whole columns
import pandas as pd  # DataFrame tables
ROOT = Path.cwd().parent  # notebooks/ is inside q2_own_scenario
sys.path.insert(0, str(ROOT / "src"))  # make src/rain_pipeline.py importable
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)  # show all columns
RAW_DIR = ROOT / "data" / "raw" / "gsod"  # original NOAA files (never edited)
import rain_pipeline as rp  # our reusable functions

### 2.3 Acquire the data
42 yearly CSV files were downloaded from `https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/<year>/<station>.csv`
for 4 stations x 2015-2025 and saved **unchanged** in `data/raw/gsod/` (2023 is missing at NOAA for Gisenyi and Ruhengeri).

In [2]:
files = sorted(RAW_DIR.glob("*.csv"))  # list the raw files
print(len(files), "files, e.g.", [f.name for f in files[:3]])  # how many and a few names
raw = rp.load_raw(RAW_DIR)  # read and stack them (all values as text)
print("Shape:", raw.shape)  # rows, columns
raw.head()  # first records

42 files, e.g. ['2015_64380099999.csv', '2015_64381099999.csv', '2015_64383099999.csv']
Shape: (10614, 29)


,STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,TEMP,TEMP_ATTRIBUTES,DEWP,DEWP_ATTRIBUTES,SLP,SLP_ATTRIBUTES,STP,STP_ATTRIBUTES,VISIB,VISIB_ATTRIBUTES,WDSP,WDSP_ATTRIBUTES,MXSPD,GUST,MAX,MAX_ATTRIBUTES,MIN,MIN_ATTRIBUTES,PRCP,PRCP_ATTRIBUTES,SNDP,FRSHTT,source_file
0,64380099999,2015-01-06,-2.4666666,28.9166666,1587.4,"KAMEMBE, RW",70.0,6,59.2,6,9999.9,0,999.9,0,6.7,6,6.0,6,8.0,999.9,75.2,*,64.4,*,0.00,I,999.9,000010,2015_64380099999.csv
1,64380099999,2015-01-12,-2.4666666,28.9166666,1587.4,"KAMEMBE, RW",71.8,9,59.9,9,9999.9,0,999.9,0,6.2,9,6.3,9,8.0,999.9,75.2,*,64.4,*,0.00,I,999.9,000010,2015_64380099999.csv
2,64380099999,2015-01-13,-2.4666666,28.9166666,1587.4,"KAMEMBE, RW",68.2,4,60.4,4,9999.9,0,999.9,0,7.0,4,3.0,4,8.0,999.9,77.0,,61.7,,0.43,G,999.9,000010,2015_64380099999.csv
3,64380099999,2015-01-14,-2.4666666,28.9166666,1587.4,"KAMEMBE, RW",70.0,7,60.9,7,9999.9,0,999.9,0,6.4,7,6.7,7,9.9,999.9,77.5,,61.9,,0.01,G,999.9,000010,2015_64380099999.csv
4,64381099999,2015-01-12,-1.677203,29.258875,1548.99,"GISENYI, RW",72.9,4,60.5,4,9999.9,0,999.9,0,4.7,4,6.0,4,8.0,999.9,75.2,*,68.0,*,0.00,I,999.9,000000,2015_64381099999.csv


In [3]:
raw.tail()  # last records

,STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,TEMP,TEMP_ATTRIBUTES,DEWP,DEWP_ATTRIBUTES,SLP,SLP_ATTRIBUTES,STP,STP_ATTRIBUTES,VISIB,VISIB_ATTRIBUTES,WDSP,WDSP_ATTRIBUTES,MXSPD,GUST,MAX,MAX_ATTRIBUTES,MIN,MIN_ATTRIBUTES,PRCP,PRCP_ATTRIBUTES,SNDP,FRSHTT,source_file
10609,64387099999,2025-08-20,-1.968628,30.13945,1481.02,"KIGALI INTERNATIONAL, RW",69.8,24,56.7,24,9999.9,0,999.9,0,6.2,24,4.3,24,8.9,999.9,78.8,*,62.4,,0.01,G,999.9,000000,2025_64387099999.csv
10610,64387099999,2025-08-21,-1.968628,30.13945,1481.02,"KIGALI INTERNATIONAL, RW",70.5,24,53.3,24,9999.9,0,999.9,0,5.7,24,4.8,24,8.0,999.9,80.6,*,60.8,*,0.00,G,999.9,000000,2025_64387099999.csv
10611,64387099999,2025-08-22,-1.968628,30.13945,1481.02,"KIGALI INTERNATIONAL, RW",69.5,24,51.0,24,9999.9,0,999.9,0,5.9,24,4.3,24,8.0,999.9,82.4,*,59.0,*,0.00,I,999.9,000000,2025_64387099999.csv
10612,64387099999,2025-08-23,-1.968628,30.13945,1481.02,"KIGALI INTERNATIONAL, RW",69.6,24,50.2,24,9999.9,0,999.9,0,6.1,24,5.1,24,11.1,999.9,80.2,,59.0,*,0.00,I,999.9,000000,2025_64387099999.csv
10613,64387099999,2025-08-24,-1.968628,30.13945,1481.02,"KIGALI INTERNATIONAL, RW",71.8,20,51.2,20,9999.9,0,999.9,0,6.1,20,5.2,20,11.1,999.9,81.7,,62.8,,0.00,I,999.9,000000,2025_64387099999.csv


In [4]:
raw.info()  # every column is text (str) because we read with dtype=str

<class 'pandas.DataFrame'>
RangeIndex: 10614 entries, 0 to 10613
Data columns (total 29 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   STATION           10614 non-null  str  
 1   DATE              10614 non-null  str  
 2   LATITUDE          10614 non-null  str  
 3   LONGITUDE         10614 non-null  str  
 4   ELEVATION         10614 non-null  str  
 5   NAME              10614 non-null  str  
 6   TEMP              10614 non-null  str  
 7   TEMP_ATTRIBUTES   10614 non-null  str  
 8   DEWP              10614 non-null  str  
 9   DEWP_ATTRIBUTES   10614 non-null  str  
 10  SLP               10614 non-null  str  
 11  SLP_ATTRIBUTES    10614 non-null  str  
 12  STP               10614 non-null  str  
 13  STP_ATTRIBUTES    10614 non-null  str  
 14  VISIB             10614 non-null  str  
 15  VISIB_ATTRIBUTES  10614 non-null  str  
 16  WDSP              10614 non-null  str  
 17  WDSP_ATTRIBUTES   10614 non-null  str  
 1

In [5]:
check = pd.DataFrame({  # requirement -> evidence
    "requirement": [">= 1,000 rows", ">= 8 columns", "numeric + categorical columns", "target can be built",
                    "4 stations", "period"],
    "evidence": [len(raw), raw.shape[1], "TEMP, PRCP ... numeric; NAME, FRSHTT categorical", "FRSHTT rain flag + DATE",
                 raw["NAME"].str.strip().unique().tolist(), f"{raw['DATE'].min()} to {raw['DATE'].max()}"],
})
check  # show

,requirement,evidence
0,">= 1,000 rows",10614
1,>= 8 columns,29
2,numeric + categorical columns,"TEMP, PRCP ... numeric; NAME, FRSHTT categorical"
3,target can be built,FRSHTT rain flag + DATE
4,4 stations,"[KAMEMBE, RW, GISENYI, RW, RUHENGERI, RW, KIGA..."
5,period,2015-01-01 to 2025-08-24


**Conclusion:** the data meets the requirements (10,614 rows, 28 columns + file name, numeric and categorical, 2015-2025).

### 2.4 Data dictionary (raw NOAA columns) - saved to `docs/data_dictionary.csv`

In [6]:
dictionary = pd.DataFrame([  # column, meaning, type, unit, notes
    ("STATION", "NOAA station code", "text (ID)", "-", "64387099999 Kigali, 64380099999 Kamembe, 64381099999 Gisenyi, 64383099999 Ruhengeri"),
    ("DATE", "Day of the observation", "date", "YYYY-MM-DD", "2015-01-01 to 2025-08-24"),
    ("LATITUDE, LONGITUDE", "Station position", "float", "degrees", "constant per station"),
    ("ELEVATION", "Station height", "float", "m", "1,481-1,856"),
    ("NAME", "Station name", "category", "-", "e.g. 'KIGALI INTERNATIONAL, RW'"),
    ("TEMP", "Mean temperature", "float", "deg F", "9999.9 = missing"),
    ("DEWP", "Mean dew point", "float", "deg F", "9999.9 = missing"),
    ("SLP", "Sea-level pressure", "float", "mbar", "9999.9 = missing"),
    ("STP", "Station pressure", "float", "mbar", "999.9 = missing"),
    ("VISIB", "Mean visibility", "float", "miles", "999.9 = missing"),
    ("WDSP", "Mean wind speed", "float", "knots", "999.9 = missing"),
    ("MXSPD", "Max sustained wind", "float", "knots", "999.9 = missing"),
    ("GUST", "Max wind gust", "float", "knots", "999.9 = missing"),
    ("MAX, MIN", "Max / min temperature", "float", "deg F", "9999.9 = missing; '*' = derived from hourly data"),
    ("PRCP", "Total precipitation", "float", "inches", "99.99 = missing"),
    ("PRCP_ATTRIBUTES", "Precipitation flag", "category", "-", "A-H = report period, I = incomplete"),
    ("SNDP", "Snow depth", "float", "inches", "999.9 = missing (no snow in Rwanda)"),
    ("FRSHTT", "Weather events", "6-digit code", "-", "Fog Rain Snow Hail Thunder Tornado, 1 = occurred"),
    ("*_ATTRIBUTES", "Number of reports used for the daily value", "integer", "count", "quality indicator"),
    ("rain_tomorrow (created)", "TARGET: rain flag of the next day", "0/1", "-", "built in Stage 4"),
], columns=["column", "meaning", "data_type", "unit", "allowed_values_notes"])
(ROOT / "docs").mkdir(exist_ok=True)  # make sure docs/ exists
dictionary.to_csv(ROOT / "docs" / "data_dictionary.csv", index=False)  # save
dictionary  # display

,column,meaning,data_type,unit,allowed_values_notes
0,STATION,NOAA station code,text (ID),-,"64387099999 Kigali, 64380099999 Kamembe, 64381..."
1,DATE,Day of the observation,date,YYYY-MM-DD,2015-01-01 to 2025-08-24
2,"LATITUDE, LONGITUDE",Station position,float,degrees,constant per station
3,ELEVATION,Station height,float,m,"1,481-1,856"
4,NAME,Station name,category,-,"e.g. 'KIGALI INTERNATIONAL, RW'"
5,TEMP,Mean temperature,float,deg F,9999.9 = missing
6,DEWP,Mean dew point,float,deg F,9999.9 = missing
7,SLP,Sea-level pressure,float,mbar,9999.9 = missing
8,STP,Station pressure,float,mbar,999.9 = missing
9,VISIB,Mean visibility,float,miles,999.9 = missing


### 2.5 Source log - saved to `docs/source_log.csv`

In [7]:
source_log = pd.DataFrame([{  # where the data came from
    "source": "NOAA National Centers for Environmental Information - Global Summary of the Day (GSOD)",
    "link": "https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/ (station list: /pub/data/noaa/isd-history.csv)",
    "date_accessed": "2026-11-11",
    "licence": "free and open from NOAA; non-US stations under WMO Resolution 40 (free for education/research, no commercial resale)",
    "collected_by": "Rwanda airport weather stations (Meteo Rwanda / airport met offices), exchanged via WMO, summarised by NOAA",
    "limitations": "only 4 stations (not every district); days missing; precipitation totals often incomplete; 2023 missing for 2 stations",
    "ethical_issues": "no personal data; bias: stations at airports (valleys/plains), not on the steep slopes where landslides happen",
}])
source_log.to_csv(ROOT / "docs" / "source_log.csv", index=False)  # save
source_log.T  # display vertically

,0
source,NOAA National Centers for Environmental Inform...
link,https://www.ncei.noaa.gov/data/global-summary-...
date_accessed,2026-11-11
licence,free and open from NOAA; non-US stations under...
collected_by,Rwanda airport weather stations (Meteo Rwanda ...
limitations,only 4 stations (not every district); days mis...
ethical_issues,no personal data; bias: stations at airports (...
